## Сегментация пользователей для персонализированных маркетинговых акций

Анализ клиентской базы интернет-магазина с помощью SQL.

**Цель:** формирование сегментов пользователей для выбора подходящей маркетинговой коммуникации в зависимости от попадания в ту или иную группу.

**Стек:** PostgreSQL, SQL, Redash

**Задача анализа:** разделить клиентов на несколько сегментов в зависимости от их покупательской активности и давности регистрации.

Сегментация должна помочь определить подходящую стратегию маркетинговых акций:

- удержание и развитие отношений с активными (постоянными) клиентами;
- реактивация клиентов с небольшим количеством покупок (разовые клиенты);
- возврат давно зарегистрированных, но не совершавших покупок клиентов (неактивных);
- приветственная коммуникация для новых пользователей.

В качестве текущей даты для расчетов используется **31 марта 2024 года**.

Описание структуры используемых данных в файле README проекта.

### SQL-запрос и логика расчетов

**Подготовка данных.** 
1. Сначала агрегируем информацию по доставленным заказам - `orders_agg`.

Для каждого клиента рассчитываем:

- количество доставленных заказов;
- среднее время доставки в днях.

В расчет включил только заказы со статусом `Delivered` (доставленные).

2. Затем агрегируем пользователей по количеству покупок - `actions_agg`.

In [2]:
WITH 
orders_agg AS (SELECT customer_id,
               COUNT(order_id) AS orders_count,
               AVG(EXTRACT(EPOCH FROM (order_delivered_customer_time - order_created_time)) / 86400.0) AS avg_delivery_days
               FROM orders
               WHERE order_status = 'Delivered'
               GROUP BY customer_id),
actions_agg AS (SELECT customer_id,
               COUNT(event_type) FILTER (WHERE event_type = 'Purchase') AS purchase_events_count
               FROM customer_actions
               GROUP BY customer_id)

IndentationError: unindent does not match any outer indentation level (<string>, line 20)

#### Формирование сегментов

После подготовки агрегатов объединяем их с основной таблицей клиентов.

`LEFT JOIN` используется для сохранения клиентов, у которых отсутствуют доставленные заказы или события покупки. 
Для таких клиентов значения количества заказов и покупок заменяются на нули с помощью `COALESCE`.

После объединения данных сегмент клиента определяется с помощью `CASE`.

In [4]:
SELECT c.customer_id,
       c.customer_city,
       c.created_at::DATE AS registration_date,
       DATE'2024-03-31' - c.created_at::DATE AS days_since_registration,
       COALESCE(o.orders_count, 0) AS orders_count,
       COALESCE(a.purchase_events_count, 0) AS purchase_events_count,
       o.avg_delivery_days,
    CASE
        WHEN COALESCE(o.orders_count, 0) >= 3 THEN 'Постоянный'
        WHEN COALESCE(o.orders_count, 0) IN (1, 2) THEN 'Разовый'
        WHEN COALESCE(o.orders_count, 0) = 0 AND (DATE '2024-03-31' - c.created_at::DATE) > 30 THEN 'Неактивный'
        WHEN COALESCE(o.orders_count, 0) = 0 AND (DATE '2024-03-31' - c.created_at::DATE) <= 30 THEN 'Новый'
    END AS segment
FROM customers c
LEFT JOIN orders_agg o USING (customer_id)
LEFT JOIN actions_agg a USING (customer_id)

IndentationError: unindent does not match any outer indentation level (<string>, line 8)

#### Результаты

В результате для каждого клиента сформирован свой маркетинговый сегмент.

Распределение клиентов по сегментам позволит оценить структуру клиентской базы и определить потенциальный объем аудитории для различных маркетинговых
сценариев.